# Módulo 1 · Clase 2 (Práctica) — Construyendo redes desde cero en PyTorch
### Deep Learning · Laboratorio

> **Para el ayudante:** este notebook es para 3 horas de laboratorio guiado. El patrón es: *explico una pieza → la corremos juntos → los estudiantes resuelven un `TODO`*. Cada ejercicio tiene su **solución** en una celda marcada `# @title Solución` (en Colab queda plegada). Sugerencia: que trabajen en parejas.

**Objetivos.** Al terminar, cada estudiante habrá:
1. Manipulado tensores y entendido `autograd` en PyTorch.
2. Implementado regresión logística **desde cero** (forward, pérdida, gradiente manual).
3. Construido y entrenado un **MLP** con `nn.Module` y un *training loop* completo.
4. Entrenado sobre un dataset real (**FashionMNIST**) y leído curvas de aprendizaje.
5. Experimentado con optimizadores, learning rate y regularización.

**Agenda (≈ 3 h):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Setup y GPU | 10 |
| 1 | Tensores y operaciones | 25 |
| 2 | Autograd | 25 |
| 3 | Regresión logística desde cero | 35 |
| — | *Descanso* | 10 |
| 4 | MLP + training loop (FashionMNIST) | 45 |
| 5 | Experimentos | 30 |
| 6 | Mini-reto | 20 |


In [ ]:
# Bloque 0 · Setup
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
# En Colab: Entorno de ejecución -> Cambiar tipo -> GPU

## Bloque 1 · Tensores y operaciones

Los **tensores** son arreglos n-dimensionales (como `numpy`) que además pueden vivir en GPU y registrar gradientes. Son la estructura de datos central de PyTorch.

In [ ]:
# Creación e inspección
a = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print("tensor:\n", a)
print("shape:", a.shape, "| dtype:", a.dtype, "| dim:", a.ndim)

# Operaciones vectorizadas (sin loops)
print("a * 2:\n", a * 2)
print("suma por columnas:", a.sum(dim=0))
print("media por filas:", a.mean(dim=1))

# Multiplicación de matrices
W = torch.randn(3, 2)
print("a @ W ->", (a @ W).shape)  # (2,3)@(3,2) = (2,2)

**Broadcasting:** PyTorch alinea automáticamente dimensiones compatibles, evitando loops explícitos. Es clave para escribir código eficiente.

In [ ]:
# Broadcasting: sumar un vector a cada fila de una matriz
M = torch.ones(3, 4)
v = torch.tensor([10., 20., 30., 40.])
print((M + v))  # v se "expande" a cada fila

### 🧩 Ejercicio 1
Dado un tensor `X` de forma `(100, 5)` (100 ejemplos, 5 features), **normaliza cada columna** para que tenga media 0 y desviación estándar 1, usando operaciones vectorizadas (sin loops).

In [ ]:
# TODO: normalización por columna (z-score)
X = torch.randn(100, 5) * 3 + 7   # datos con media ~7 y std ~3

# X_norm = ...
# print("media por columna:", X_norm.mean(dim=0))
# print("std por columna:", X_norm.std(dim=0))

In [ ]:
# @title Solución
X_norm = (X - X.mean(dim=0)) / X.std(dim=0)
print("media por columna:", torch.round(X_norm.mean(dim=0), decimals=4))
print("std por columna:", torch.round(X_norm.std(dim=0), decimals=4))

## Bloque 2 · Autograd

PyTorch construye un **grafo computacional** dinámicamente y, al llamar `.backward()`, calcula todos los gradientes por backpropagation. Solo hay que marcar los tensores con `requires_grad=True`.

In [ ]:
# Ejemplo: minimizar f(x) = (x - 3)^2, cuyo mínimo está en x=3
x = torch.tensor([0.0], requires_grad=True)
lr = 0.1
for step in range(30):
    loss = (x - 3)**2
    loss.backward()                 # calcula dloss/dx
    with torch.no_grad():           # actualizar sin registrar en el grafo
        x -= lr * x.grad
    x.grad.zero_()                  # importante: limpiar gradientes
print("x final:", round(x.item(), 4), "(esperado ~3.0)")

**Por qué `x.grad.zero_()`:** PyTorch *acumula* gradientes. Si no los limpias en cada paso, se suman los de pasos anteriores. En un training loop esto se hace con `optimizer.zero_grad()`.

### 🧩 Ejercicio 2
Usa autograd para encontrar el mínimo de $f(x, y) = (x-2)^2 + (y+1)^2$. Empieza en `(0, 0)` y reporta `(x, y)` tras 50 pasos.

In [ ]:
# TODO
p = torch.tensor([0.0, 0.0], requires_grad=True)
# loop de gradiente descendente ...
# print(p.detach())

In [ ]:
# @title Solución
p = torch.tensor([0.0, 0.0], requires_grad=True)
lr = 0.1
for _ in range(50):
    loss = (p[0]-2)**2 + (p[1]+1)**2
    loss.backward()
    with torch.no_grad():
        p -= lr * p.grad
    p.grad.zero_()
print("mínimo encontrado:", torch.round(p.detach(), decimals=3), "(esperado ~[2, -1])")

## Bloque 3 · Regresión logística desde cero

Antes de usar las clases de alto nivel, implementemos un clasificador **manualmente** para ver toda la maquinaria. Regresión logística = una neurona con activación sigmoid + pérdida de entropía cruzada binaria.

$$ \hat{y} = \sigma(\mathbf{w}^\top \mathbf{x} + b), \qquad \mathcal{L} = -\frac{1}{N}\sum_i \big[y_i \log \hat y_i + (1-y_i)\log(1-\hat y_i)\big] $$

Generamos dos nubes de puntos (2 clases) y entrenamos.

In [ ]:
# Datos sintéticos: dos clases gaussianas
torch.manual_seed(0)
n = 200
c0 = torch.randn(n, 2) + torch.tensor([-2., -2.])
c1 = torch.randn(n, 2) + torch.tensor([ 2.,  2.])
X = torch.cat([c0, c1]); y = torch.cat([torch.zeros(n), torch.ones(n)])

plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', alpha=0.5, s=12)
plt.title("Dataset: 2 clases"); plt.grid(True); plt.show()

In [ ]:
# Entrenamiento manual (sin optimizer, gradiente calculado por autograd)
w = torch.zeros(2, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1
hist = []
for epoch in range(200):
    z = X @ w + b
    y_hat = torch.sigmoid(z)
    loss = F.binary_cross_entropy(y_hat, y)
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad; b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()
    hist.append(loss.item())

acc = ((torch.sigmoid(X @ w + b) > 0.5).float() == y).float().mean()
print(f"loss final: {hist[-1]:.4f} | accuracy: {acc.item():.3f}")
plt.figure(figsize=(5,3)); plt.plot(hist); plt.xlabel("época"); plt.ylabel("loss"); plt.title("Curva de entrenamiento"); plt.grid(True); plt.show()

### 🧩 Ejercicio 3
Visualiza la **frontera de decisión** aprendida. Pista: evalúa el modelo en una grilla de puntos y usa `plt.contourf`.

In [ ]:
# @title Solución
xx, yy = torch.meshgrid(torch.linspace(-6,6,200), torch.linspace(-6,6,200), indexing='xy')
grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)
with torch.no_grad():
    zz = torch.sigmoid(grid @ w + b).reshape(xx.shape)
plt.figure(figsize=(4.5,4))
plt.contourf(xx, yy, zz, levels=20, cmap='bwr', alpha=0.6)
plt.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolor='k', s=12)
plt.title("Frontera de decisión"); plt.show()

## Bloque 4 · MLP + training loop sobre FashionMNIST

Ahora el flujo **profesional**: definimos la red con `nn.Module`, usamos un `DataLoader` para iterar en mini-batches, un `optimizer` y un `loss` de la librería. Dataset: **FashionMNIST** (28×28 px, 10 clases de ropa).

In [ ]:
# Carga de datos (se descarga automáticamente en Colab)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([transforms.ToTensor(),
                          transforms.Normalize((0.5,), (0.5,))])
train_ds = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tfm)
test_ds  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tfm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256)

classes = ['Camiseta','Pantalón','Suéter','Vestido','Abrigo','Sandalia','Camisa','Zapatilla','Bolso','Botín']
print("train:", len(train_ds), "| test:", len(test_ds))

# Visualizar algunas muestras
imgs, labels = next(iter(train_loader))
fig, ax = plt.subplots(1,6, figsize=(12,2.2))
for i in range(6):
    ax[i].imshow(imgs[i,0], cmap='gray'); ax[i].set_title(classes[labels[i]]); ax[i].axis('off')
plt.show()

In [ ]:
# Definición del MLP
class MLP(nn.Module):
    def __init__(self, hidden=256, p_drop=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),               # (B,1,28,28) -> (B,784)
            nn.Linear(28*28, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),  nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, 10)       # logits de 10 clases
        )
    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
print("parámetros:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Funciones de entrenamiento y evaluación reutilizables
def evaluate(model, loader):
    model.eval(); correct = total = 0; loss_sum = 0.0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss_sum += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum/total, correct/total

def train(model, epochs=5, lr=1e-3, opt_name="adam", weight_decay=0.0):
    loss_fn = nn.CrossEntropyLoss()
    if opt_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    hist = {"train_loss":[], "val_loss":[], "val_acc":[]}
    for ep in range(epochs):
        model.train(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward(); opt.step()
            running += loss.item()*len(y)
        tr = running/len(train_loader.dataset)
        vl, va = evaluate(model, test_loader)
        hist["train_loss"].append(tr); hist["val_loss"].append(vl); hist["val_acc"].append(va)
        print(f"época {ep+1}/{epochs} | train_loss {tr:.3f} | val_loss {vl:.3f} | val_acc {va:.3f}")
    return hist

In [ ]:
# Entrenar el MLP base (~5 épocas, rápido en GPU)
model = MLP(hidden=256, p_drop=0.0).to(device)
hist = train(model, epochs=5, lr=1e-3, opt_name="adam")

In [ ]:
# Curvas de aprendizaje
fig, ax = plt.subplots(1,2, figsize=(11,3.5))
ax[0].plot(hist["train_loss"], label="train"); ax[0].plot(hist["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("época"); ax[0].legend(); ax[0].grid(True)
ax[1].plot(hist["val_acc"]); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("época"); ax[1].grid(True)
plt.show()

## Bloque 5 · Experimentos

Aquí está el corazón de la práctica: cambiar **una cosa a la vez** y observar el efecto. Corre cada experimento y comenta los resultados con los estudiantes.

### Experimento A — Optimizador: SGD vs Adam

In [ ]:
for opt_name in ["sgd", "adam"]:
    print(f"\n=== {opt_name.upper()} ===")
    m = MLP().to(device)
    train(m, epochs=3, lr=1e-3, opt_name=opt_name)

### Experimento B — Learning rate
Prueba un LR muy chico, uno razonable y uno muy grande. Observa cuál no aprende y cuál diverge.

In [ ]:
for lr in [1e-4, 1e-3, 1e-1]:
    print(f"\n=== lr = {lr} ===")
    m = MLP().to(device)
    train(m, epochs=3, lr=lr, opt_name="adam")

### 🧩 Experimento C — Regularización contra overfitting
Entrena **muchas épocas sin regularización** y luego **con dropout + weight decay**. Compara las curvas de validación.

In [ ]:
# @title Solución
print("Sin regularización:")
m1 = MLP(p_drop=0.0).to(device)
h1 = train(m1, epochs=8, lr=1e-3, weight_decay=0.0)

print("\nCon dropout=0.3 y weight_decay=1e-4:")
m2 = MLP(p_drop=0.3).to(device)
h2 = train(m2, epochs=8, lr=1e-3, weight_decay=1e-4)

plt.figure(figsize=(6,3.5))
plt.plot(h1["val_loss"], label="sin reg.")
plt.plot(h2["val_loss"], label="con reg.")
plt.xlabel("época"); plt.ylabel("val loss"); plt.legend(); plt.title("Efecto de la regularización"); plt.grid(True); plt.show()

## Bloque 6 · 🏁 Mini-reto

Pon a prueba lo aprendido. **Objetivo: superar 89% de accuracy en el test de FashionMNIST.**

Ideas para mejorar:
- Más capas o más neuronas.
- Ajustar dropout y weight decay.
- Más épocas + learning rate adecuado.
- Probar `BatchNorm1d` entre capas.

Define tu arquitectura, entrénala y evalúa. ¡Que compitan por la mejor accuracy!

In [ ]:
# TODO: tu mejor modelo
class MiRed(nn.Module):
    def __init__(self):
        super().__init__()
        # define tus capas aquí
        ...
    def forward(self, x):
        ...

# mi_modelo = MiRed().to(device)
# train(mi_modelo, epochs=..., lr=...)
# print("Test accuracy:", evaluate(mi_modelo, test_loader)[1])

In [ ]:
# @title Solución de referencia (MLP con BatchNorm)
class RedConBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x): return self.net(x)

best = RedConBN().to(device)
train(best, epochs=12, lr=1e-3, opt_name="adam", weight_decay=1e-4)
print("\nTest accuracy final:", round(evaluate(best, test_loader)[1], 4))

## Cierre

Hoy implementaste, desde los tensores hasta un MLP completo, **toda la maquinaria de entrenamiento**: forward, pérdida, backward, actualización, y el efecto real de los hiperparámetros.

**En el Módulo 2** veremos por qué un MLP no es ideal para imágenes y cómo las **redes convolucionales** explotan la estructura espacial — y luego los **Vision Transformers**.